
# ZynNova ZynMorph：电极生成 + 微结构表征/重建 + COMSOL round-trip 集成测试

这个 notebook 验证本次源码级整合后的三条主链：

1. **COMSOL MPHTXT 可逆材料域映射**：ZynNova 原始 phase ID 可以包含 `0`；写入 COMSOL 时转换为正整数 entity，读回时自动恢复原始材料 ID。
2. **粒子电极生成**：球/椭球/混合颗粒、RSA/重力/伪重力/伪静电 packing、contained/extended/periodic 边界、PSD、CBD、纳米孔隙、padding/cropping、通道、单颗粒 tracking、HDF5/VTK。
3. **descriptor-space microstructure engine**：完整 descriptor/loss/optimizer 注册表、binary single-phase / multi-phase、3D directional slice、directional merge、interpolation、gradient / Yeong–Torquato、coarse→fine multigrid reconstruction。

最终任何生成或重建后的复杂体素域都可以继续进入：

```text
MicrostructureVolume
    -> mesh_complex_regions(...)
    -> conforming multi-material PLC
    -> native TetGen C++
    -> irregular adaptive Tet4
    -> COMSOL MPHTXT
```

> 本 notebook 不依赖外部的 MCS-CICE 或 MCRpy Python 包。所有调用均为 `zynnova.zynmorph` 自己的 API。


In [ ]:

from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import zynnova
except ModuleNotFoundError:
    candidates = [Path.cwd(), *Path.cwd().parents]
    explicit = os.environ.get('ZYNNOVA_PROJECT_ROOT')
    if explicit:
        candidates = [Path(explicit), *candidates]
    root = next((p for p in candidates if (p/'src'/'zynnova').is_dir()), None)
    if root is None:
        raise
    sys.path.insert(0, str(root/'src'))
    import zynnova

from zynnova.geometry import VolumeMesh
from zynnova.zynmorph import (
    CBDSettings,
    ChannelSettings,
    CharacterizationSettings,
    ElectrodeComposition,
    ElectrodeSynthesisConfig,
    IrregularMeshPolicy,
    PackingSettings,
    ParticleDistribution,
    ReconstructionSettings,
    characterize,
    export_comsol_mphtxt,
    generate_particle_electrode,
    interpolate,
    load_comsol_tet4_mphtxt,
    match,
    merge_directional,
    mesh_complex_regions,
    reconstruct,
    tetgen_native_status,
    translate_microstructure,
)
from zynnova.zynmorph.microstructure import DESCRIPTORS, LOSSES, OPTIMIZERS

RUN_NATIVE_TETGEN = os.environ.get('ZYNNOVA_RUN_NATIVE_TETGEN', '1') == '1'
OUTPUT = Path('zynnova_runs') / 'mcs_mcr_comsol_integration'
OUTPUT.mkdir(parents=True, exist_ok=True)

print('ZynNova:', zynnova.__file__)
print('Python :', sys.executable)
print('Native TetGen requested:', RUN_NATIVE_TETGEN)


## 1. COMSOL：原始材料 ID 0..4 的严格 round-trip

In [ ]:

nodes=[]
tets=[]
for region in range(5):
    x=2.0*region
    start=len(nodes)
    nodes.extend([(x,0,0),(x+1,0,0),(x,1,0),(x,0,1)])
    tets.append((start,start+1,start+2,start+3))
mesh = VolumeMesh(
    np.asarray(nodes, dtype=np.float64),
    np.asarray(tets, dtype=np.int64),
    np.arange(5, dtype=np.int32),
    {i: f'phase_{i}' for i in range(5)},
)
path = OUTPUT/'roundtrip_0_to_4.mphtxt'
report = export_comsol_mphtxt(path, mesh, include_boundaries=False)
roundtrip = load_comsol_tet4_mphtxt(path)

print('Writer region -> COMSOL entity:', report.region_entity_map)
print('Original regions:', np.unique(mesh.cell_regions))
print('Read-back regions:', np.unique(roundtrip.cell_regions))
print('Reader metadata:', roundtrip.metadata)

assert report.region_entity_map == {0:1,1:2,2:3,3:4,4:5}
assert np.array_equal(roundtrip.cell_regions, mesh.cell_regions)
assert roundtrip.region_names == mesh.region_names
assert roundtrip.metadata['restored_original_regions'] is True
print('COMSOL reversible region mapping: PASS')


## 2. 完整插件注册表

In [ ]:

plugin_table = pd.DataFrame({
    'descriptors': pd.Series(DESCRIPTORS.names()),
    'losses': pd.Series(LOSSES.names()),
    'optimizers': pd.Series(OPTIMIZERS.names()),
})
display(plugin_table)

expected_descriptors = {
    'VolumeFractions','Variation','FFTCorrelations','TwoPointCorrelations',
    'Correlations','FFTCrossCorrelations','CrossCorrelations','LineCorrelations',
    'LinealPath','LinealPathApproximation','LineLinealPathApproximation',
    'GramMatrices','MultiPhaseGramMatrices','OrientationDescriptor'
}
expected_losses = {'MSE','SSE','RMS','L1','L2'}
expected_optimizers = {
    'Adam','Adamax','Nadam','RMSprop','SGD','Adagrad','Adadelta',
    'LBFGSB','TNC','SimulatedAnnealing','YTPost'
}
assert expected_descriptors <= set(DESCRIPTORS.names())
assert expected_losses <= set(LOSSES.names())
assert expected_optimizers <= set(OPTIMIZERS.names())


## 3. 生成一个复杂混合颗粒电极（含 padding/cropping、CBD、通道和单颗粒 tracking）

In [ ]:

composition = ElectrodeComposition(
    active_mass_fraction=0.92,
    carbon_mass_fraction=0.04,
    binder_mass_fraction=0.04,
    active_density_g_cm3=2.26,
    carbon_density_g_cm3=1.85,
    binder_density_g_cm3=1.60,
    active_specific_capacity_mAh_g=372.0,
)

config = ElectrodeSynthesisConfig(
    shape_zyx=(24, 28, 30),
    voxel_size_m=0.75e-6,
    active_volume_fraction=0.55,
    seed=20260818,
    padding_voxels=(3,3,4),
    crop_after_generation=True,
    particle_distribution=ParticleDistribution(
        geometry='mixed',
        median_diameter_vox=6.0,
        lognormal_sigma=0.22,
        minimum_diameter_vox=3.0,
        maximum_diameter_vox=10.0,
        sphere_fraction=0.40,
        axis_ratio_ranges=((0.60,1.45),(0.60,1.45)),
        angle_tolerance_degrees=30.0,
    ),
    packing=PackingSettings(
        method='rsa',
        boundary_mode='extended',
        overlap_fraction=0.12,
        max_attempts_per_particle=500,
    ),
    cbd=CBDSettings(
        method='mixed',
        target_volume_fraction=0.08,
        nanoporosity=0.35,
        correlation_length_vox=2.0,
        mixed_bridge_fraction=0.55,
    ),
    channel=ChannelSettings(enabled=True, radius_vox=1.7, phase=0),
    individual_particle_labels=True,
    composition=composition,
)

electrode = generate_particle_electrode(config)
print(electrode.statistics)
print(electrode.psd_validation)
print('Final shape:', electrode.volume.labels.shape)
print('Packing shape:', electrode.volume.metadata['packing_shape_zyx'])
print('Surviving particle records:', len(electrode.particles))
assert electrode.volume.labels.shape == config.shape_zyx
# Channel carving intentionally converts any material inside the tunnel to
# electrolyte, so the *post-channel* active fraction can only decrease.
assert electrode.statistics.active_fraction <= config.active_volume_fraction + 1/np.prod(config.shape_zyx)
assert electrode.statistics.active_fraction > config.active_volume_fraction - 0.03

exports = electrode.export(OUTPUT/'electrode', formats=('npz','h5','vtk'))
print(exports)


In [ ]:

labels = electrode.volume.labels
fig, ax = plt.subplots(1, 3, figsize=(15,4))
ax[0].imshow(labels[labels.shape[0]//2], cmap='tab10', origin='lower')
ax[0].set_title('phase slice z')
ax[1].imshow(labels[:, labels.shape[1]//2, :], cmap='tab10', origin='lower', aspect='auto')
ax[1].set_title('phase slice y')
tracking = electrode.particle_labels[electrode.particle_labels.shape[0]//2]
ax[2].imshow(np.where(tracking>=0, tracking, np.nan), cmap='tab20', origin='lower')
ax[2].set_title('individual particle tracking')
plt.tight_layout(); plt.show()


## 4. MCR 风格周期平移

验证周期 translation 不改变相体积分数、数组尺寸和相编号。随后章节顺延。

In [ ]:
translated = translate_microstructure(electrode.volume.labels, (3, -5, 7))
restored = translate_microstructure(translated, (-3, 5, -7))
assert translated.shape == electrode.volume.labels.shape
assert np.array_equal(np.bincount(translated.ravel()), np.bincount(electrode.volume.labels.ravel()))
assert np.array_equal(restored, electrode.volume.labels)
print("Periodic translation: PASS")

## 4. MCR 风格：3D directional characterization + multigrid descriptor

In [ ]:

char_settings = CharacterizationSettings(
    descriptor_types=('VolumeFractions','Variation','Correlations','GramMatrices'),
    limit_to=4,
    use_multiphase=True,
    use_multigrid_descriptors=True,
    multigrid_levels=2,
    slice_mode='average',
    isotropic=False,
    phase_ids=tuple(sorted(map(int, np.unique(labels)))),
)
char3d = characterize(labels, char_settings)
for name, result in char3d.descriptors.items():
    print(name, np.asarray(result.values).shape, result.metadata)
    assert np.all(np.isfinite(result.values))


## 5. Binary single-phase 模式：只表征 phase 1，但重建仍严格满足两相和为 1

In [ ]:

y,x=np.indices((16,16))
binary=((x//4+y//4)%2).astype(np.int32)
single_target=characterize(binary, CharacterizationSettings(
    descriptor_types=('VolumeFractions','Variation'),
    use_multiphase=False,
    use_multigrid_descriptors=False,
    limit_to=4,
))
print('Stored descriptor phase channels:', single_target['VolumeFractions'].values)
print('Full volume fractions in metadata:', single_target.metadata['volume_fractions'])

single_result=reconstruct(single_target, settings=ReconstructionSettings(
    descriptor_types=('VolumeFractions','Variation'),
    descriptor_weights=(1.0,2.0),
    optimizer_type='Adam',
    use_multiphase=False,
    use_multigrid_descriptors=False,
    max_iter=6,
    learning_rate=0.05,
    dtype='float64',
))
assert np.allclose(single_result.probabilities.sum(axis=0),1.0,atol=1e-10)
print('single-phase reconstruction final loss:', single_result.final_loss)


## 6. 2D→3D directional merge + coarse→fine multigrid reconstruction

In [ ]:

slice_a = binary
slice_b = np.roll(binary, 2, axis=0)
cs = CharacterizationSettings(
    descriptor_types=('VolumeFractions','Variation'),
    use_multigrid_descriptors=True,
    multigrid_levels=2,
    limit_to=4,
)
ca=characterize(slice_a, cs)
cb=characterize(slice_b, cs)
directional=merge_directional((ca,cb))  # source convention: (A,A,B)
print('directional merge metadata:', directional.metadata)
for name,result in directional.descriptors.items():
    print(name, np.asarray(result.values).shape)
    assert np.asarray(result.values).shape[0] == 3

multi_result = reconstruct(
    directional,
    desired_shape=(16,16,16),
    settings=ReconstructionSettings(
        descriptor_types=('VolumeFractions','Variation'),
        descriptor_weights=(1.0,2.0),
        optimizer_type='Adam',
        use_multigrid_descriptors=True,
        use_multigrid_reconstruction=True,
        multigrid_levels=2,
        slice_mode='average',
        isotropic=False,
        max_iter=6,
        learning_rate=0.03,
        dtype='float64',
    ),
)
print('multigrid levels:', multi_result.metadata['multigrid_reconstruction_levels'])
print('final loss:', multi_result.final_loss)
assert multi_result.labels.shape == (16,16,16)
assert multi_result.metadata['multigrid_reconstruction_levels'] >= 2


## 7. Descriptor interpolation 与 match 快捷流程

In [ ]:

rot=np.rot90(binary)
ca2=characterize(binary, CharacterizationSettings(
    descriptor_types=('VolumeFractions','Variation'), use_multigrid_descriptors=False
))
cb2=characterize(rot, CharacterizationSettings(
    descriptor_types=('VolumeFractions','Variation'), use_multigrid_descriptors=False
))
path=interpolate(ca2,cb2,5)
print('interpolated characterizations:', len(path))
assert len(path)==5

matched_target, matched = match(
    binary,
    characterization_settings=CharacterizationSettings(
        descriptor_types=('VolumeFractions','Variation'),
        use_multigrid_descriptors=False,
    ),
    reconstruction_settings=ReconstructionSettings(
        descriptor_types=('VolumeFractions','Variation'),
        descriptor_weights=(1.0,2.0),
        optimizer_type='Adam',
        use_multigrid_descriptors=False,
        max_iter=4,
        learning_rate=0.04,
        dtype='float64',
    ),
)
assert matched.labels.shape == binary.shape
print('match final loss:', matched.final_loss)


## 8. 任意复杂微结构 → native TetGen → irregular Tet4 → COMSOL（本机有 TetGen 时执行）

In [ ]:

status=tetgen_native_status()
print(status)

if RUN_NATIVE_TETGEN and status.available:
    policy=IrregularMeshPolicy(
        base_edge_length_m=1.5e-6,
        region_edge_lengths_m={
            int(config.electrolyte_phase): 1.25e-6,
            int(config.active_phase): 1.10e-6,
            int(config.cbd_phase): 0.90e-6,
        },
        smoothing_iterations=3,
        radius_edge_ratio=1.5,
        minimum_dihedral_degrees=7.0,
    )
    fem=mesh_complex_regions(
        electrode.volume,
        policy=policy,
        maximum_tetrahedra=2_000_000,
    )
    assert fem.quality.fem_ready
    mesh_path=OUTPUT/'electrode_irregular_tet.mphtxt'
    rep=export_comsol_mphtxt(mesh_path, fem.mesh)
    reread=load_comsol_tet4_mphtxt(mesh_path)
    print('TetGen nodes/tets:', reread.n_nodes, reread.n_cells)
    print('Regions:', np.unique(reread.cell_regions))
    assert set(map(int,np.unique(reread.cell_regions))) == set(map(int,np.unique(fem.mesh.cell_regions)))
else:
    print('Native TetGen step skipped in this runtime. On your Windows build leave ZYNNOVA_RUN_NATIVE_TETGEN unset.')


## 9. 最终审计摘要

In [ ]:

summary={
    'comsol_roundtrip_original_regions': list(map(int,np.unique(roundtrip.cell_regions))),
    'descriptor_plugins': list(DESCRIPTORS.names()),
    'loss_plugins': list(LOSSES.names()),
    'optimizer_plugins': list(OPTIMIZERS.names()),
    'electrode_shape': list(electrode.volume.labels.shape),
    'electrode_particles': len(electrode.particles),
    'active_fraction': electrode.statistics.active_fraction,
    'multigrid_reconstruction_levels': multi_result.metadata['multigrid_reconstruction_levels'],
    'native_tetgen_available': bool(status.available),
}
path=OUTPUT/'integration_validation.json'
path.write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))
print('Report:',path)
